<h1>Check local dataset for completion and adjust file paths</h1>

In [1]:
# Dependencies
import os
import pandas as pd

<h2>Collect local file paths for all dcm files in the dataset</h2>

In [2]:
# Function to retrieve case related information from the filepath
def parse_case_id_from_path(path):
    parts = path.replace("\\", "/").split("/")
    for part in parts[::-1]:
        for prefix in ["Calc-Test_", "Mass-Test_", "Calc-Training_", "Mass-Training_"]:
            if part.startswith(prefix):
                # Determine whether image belongs to train or test set
                if "Test_" in part:
                    dataset_split = "test"
                elif "Training_" in part:
                    dataset_split = "train"
                else:
                    dataset_split = "unknown"
                case_type = 'calc' if part.startswith("Calc") else 'mass'
                id_parts = part.replace(prefix, "").split("_")
                if len(id_parts) < 4:
                    raise ValueError(f"Unexpected format in case folder: {part}")
                patient_id = "_".join(id_parts[:2])  # example: P_00038
                side = id_parts[2].upper()  # example: LEFT or RIGHT
                angle = id_parts[3].replace("_1", "").replace("_2", "").upper()  # example: MLO or CC
                return dataset_split, case_type, patient_id, side, angle
    raise ValueError(f"No valid case ID found in path: {path}")

# Function to classify image type: full x-ray, cropped image, binary ROI mask file
def classify_files(folder_path):
    dcm_files = [f for f in os.listdir(folder_path) if f.endswith('.dcm')]
    full_paths = [os.path.join(folder_path, f) for f in dcm_files]
    if len(full_paths) == 1:
        return [(full_paths[0], 'full')]
    elif len(full_paths) == 2:
        sizes = [(f, os.path.getsize(f)) for f in full_paths]
        sizes.sort(key=lambda x: x[1])
        return [(sizes[0][0], 'binary'), (sizes[1][0], 'cropped')]
    return [(f, 'unknown') for f in full_paths]

# Main function for information collection
def collect_image_info(root_dir):
    data = []
    for root, _, files in os.walk(root_dir):
        dcm_files = [f for f in files if f.endswith('.dcm')]
        if not dcm_files:
            continue

        try:
            dataset_split, case_type, patient_id, side, angle = parse_case_id_from_path(root)
        except ValueError as e:
            print(f"Skipping folder: {e}")
            continue

        file_info = classify_files(root)
        for filepath, img_type in file_info:
            data.append({
                'split': dataset_split,
                'filepath': filepath,
                'image_type': img_type,
                'case_type': case_type,
                'patient_id': patient_id,
                'left or right breast': side,
                'image view': angle
            })
    return pd.DataFrame(data)

In [3]:
root_folder = "../data/raw/CBIS-DDSM"
local_files_df = collect_image_info(root_folder)
local_files_df.tail()

,split,filepath,image_type,case_type,patient_id,left or right breast,image view
10234,train,../data/raw/CBIS-DDSM\Mass-Training_P_02092_LE...,full,mass,P_02092,LEFT,CC
10235,train,../data/raw/CBIS-DDSM\Mass-Training_P_02092_LE...,full,mass,P_02092,LEFT,CC
10236,train,../data/raw/CBIS-DDSM\Mass-Training_P_02092_LE...,full,mass,P_02092,LEFT,MLO
10237,train,../data/raw/CBIS-DDSM\Mass-Training_P_02092_LE...,full,mass,P_02092,LEFT,MLO
10238,train,../data/raw/CBIS-DDSM\Mass-Training_P_02092_LE...,full,mass,P_02092,LEFT,MLO


<h2>Segregate local file data to calcifications and masses</h2>

In [4]:
calc_files_local = local_files_df[local_files_df["case_type"] == "calc"].copy()
mass_files_local = local_files_df[local_files_df["case_type"] == "mass"].copy()

<h2>Dataset subset: Masses</h2>

<h3>Load metadata for the mass set</h3>

In [5]:
# Load metadata CSVs
mass_test = pd.read_csv("../data/raw/CBIS-DDSM_metadata/mass_case_description_test_set.csv")
mass_train = pd.read_csv("../data/raw/CBIS-DDSM_metadata/mass_case_description_train_set.csv")

In [6]:
# Add set type (train or test) to the df
mass_train['set'] = 'train'
mass_test['set'] = 'test'

# Merge the datasets
mass_df = pd.concat([mass_train, mass_test], ignore_index=True)

<h3> Ensure that all cases are existing for masses</h3>

In [7]:
# Prepare metadata file information
df1_check = mass_df.copy()
df1_subset = df1_check[['patient_id', 'left or right breast', 'image view']]

# Prepare local file information
df2_check = mass_files_local.copy()
df2_subset = df2_check[['patient_id', 'left or right breast', 'image view']]

# Compare sets as tuples
df1_set = set(map(tuple, df1_subset.to_numpy()))
df2_set = set(map(tuple, df2_subset.to_numpy()))

# Find overlaps and differences
intersection = df1_set & df2_set
only_in_df1 = df1_set - df2_set
only_in_df2 = df2_set - df1_set

# Results
print("Matching entries:", len(intersection))
print("Only in mass_orig_merged:", len(only_in_df1))
print("Only in mass_full:", len(only_in_df2))


Matching entries: 1592
Only in mass_orig_merged: 0
Only in mass_full: 0


<h3>Update file paths for the mass subset of the CBIS-DDSM dataset</h3>

In [8]:
for _, row in mass_files_local.iterrows():
    condition = (
        (mass_df["patient_id"] == row["patient_id"]) &
        (mass_df["left or right breast"] == row["left or right breast"]) &
        (mass_df["image view"] == row["image view"])
    )

    if condition.sum() == 0:
        print("No match for:", row["patient_id"], row["left or right breast"], row["image view"])
        continue

    idx = mass_df[condition].index[0]

    if row["image_type"] == "full":
        mass_df.loc[idx, "image file path"] = row["filepath"]
    elif row["image_type"] == "cropped":
        mass_df.loc[idx, "cropped image file path"] = row["filepath"]
    elif row["image_type"] == "binary":
        mass_df.loc[idx, "ROI mask file path"] = row["filepath"]

In [9]:
mass_df.head()

,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path,set
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,../data/raw/CBIS-DDSM\Mass-Training_P_00001_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00001_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00001_LE...,train
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,../data/raw/CBIS-DDSM\Mass-Training_P_00001_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00001_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00001_LE...,train
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,../data/raw/CBIS-DDSM\Mass-Training_P_00004_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00004_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00004_LE...,train
3,P_00004,3,LEFT,MLO,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,../data/raw/CBIS-DDSM\Mass-Training_P_00004_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00004_LE...,../data/raw/CBIS-DDSM\Mass-Training_P_00004_LE...,train
4,P_00004,3,RIGHT,MLO,1,mass,OVAL,CIRCUMSCRIBED,4,BENIGN,5,../data/raw/CBIS-DDSM\Mass-Training_P_00004_RI...,../data/raw/CBIS-DDSM\Mass-Training_P_00004_RI...,../data/raw/CBIS-DDSM\Mass-Training_P_00004_RI...,train


<h3>Split dataframe and save to csv</h3>

In [10]:
mass_train = mass_df[mass_df["set"] == "train"]
test_train = mass_df[mass_df["set"] == "test"]

In [11]:
mass_train.to_csv("../data/processed/mass_training.csv", index=False)
test_train.to_csv("../data/processed/mass_test.csv", index=False)

<h2>Dataset subset: Calcifications</h2>

<h3>Load metadata for the calcifications sets</h3>

In [12]:
# Load metadata CSVs
calc_test = pd.read_csv("../data/raw/CBIS-DDSM_metadata/calc_case_description_test_set.csv")
calc_train = pd.read_csv("../data/raw/CBIS-DDSM_metadata/calc_case_description_train_set.csv")

In [13]:
# Add set type (train or test) to the df
calc_train['set'] = 'train'
calc_test['set'] = 'test'

# Merge the datasets
calc_df = pd.concat([calc_train, calc_test], ignore_index=True)
calc_df.head()

,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path,set
0,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.959...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9...,train
1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.95...,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1....,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1....,train
2,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.95...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.95...,train
3,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,Calc-Training_P_00007_LEFT_MLO/1.3.6.1.4.1.959...,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9...,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9...,train
4,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590...,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.95...,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.95...,train


In [14]:
# rename breast density column to match the masses subset
calc_df.rename(columns={'breast density': 'breast_density'}, inplace=True)

<h3> Ensure that all cases are existing for calcifications</h3>

In [15]:
# Prepare metadata file information
df1_check = calc_df.copy()
df1_subset = df1_check[['patient_id', 'left or right breast', 'image view']]

# Prepare local file information
df2_check = calc_files_local.copy()
df2_subset = df2_check[['patient_id', 'left or right breast', 'image view']]

# Compare sets as tuples
df1_set = set(map(tuple, df1_subset.to_numpy()))
df2_set = set(map(tuple, df2_subset.to_numpy()))

# Find overlaps and differences
intersection = df1_set & df2_set
only_in_df1 = df1_set - df2_set
only_in_df2 = df2_set - df1_set

# Results
print("Matching entries:", len(intersection))
print("Only in mass_orig_merged:", len(only_in_df1))
print("Only in mass_full:", len(only_in_df2))

Matching entries: 1511
Only in mass_orig_merged: 0
Only in mass_full: 0


<h3>Update file paths for the calcifications subset</h3>

In [16]:
for _, row in calc_files_local.iterrows():
    condition = (
        (calc_df["patient_id"] == row["patient_id"]) &
        (calc_df["left or right breast"] == row["left or right breast"]) &
        (calc_df["image view"] == row["image view"])
    )

    if condition.sum() == 0:
        print("No match for:", row["patient_id"], row["left or right breast"], row["image view"])
        continue

    idx = calc_df[condition].index[0]

    if row["image_type"] == "full":
        calc_df.loc[idx, "image file path"] = row["filepath"]
    elif row["image_type"] == "cropped":
        calc_df.loc[idx, "cropped image file path"] = row["filepath"]
    elif row["image_type"] == "binary":
        calc_df.loc[idx, "ROI mask file path"] = row["filepath"]

In [17]:
calc_df.head()

,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path,set
0,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,../data/raw/CBIS-DDSM\Calc-Training_P_00005_RI...,../data/raw/CBIS-DDSM\Calc-Training_P_00005_RI...,../data/raw/CBIS-DDSM\Calc-Training_P_00005_RI...,train
1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,../data/raw/CBIS-DDSM\Calc-Training_P_00005_RI...,../data/raw/CBIS-DDSM\Calc-Training_P_00005_RI...,../data/raw/CBIS-DDSM\Calc-Training_P_00005_RI...,train
2,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,../data/raw/CBIS-DDSM\Calc-Training_P_00007_LE...,../data/raw/CBIS-DDSM\Calc-Training_P_00007_LE...,../data/raw/CBIS-DDSM\Calc-Training_P_00007_LE...,train
3,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,../data/raw/CBIS-DDSM\Calc-Training_P_00007_LE...,../data/raw/CBIS-DDSM\Calc-Training_P_00007_LE...,../data/raw/CBIS-DDSM\Calc-Training_P_00007_LE...,train
4,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3,../data/raw/CBIS-DDSM\Calc-Training_P_00008_LE...,../data/raw/CBIS-DDSM\Calc-Training_P_00008_LE...,../data/raw/CBIS-DDSM\Calc-Training_P_00008_LE...,train


<h3>Split dataframe and save to csv</h3>

In [18]:
calc_train = calc_df[calc_df["set"] == "train"]
calc_train = calc_df[calc_df["set"] == "test"]

In [19]:
calc_train.to_csv("../data/processed/calc_training.csv", index=False)
calc_train.to_csv("../data/processed/calc_test.csv", index=False)